# ML Pipeline (IS455 CRISP-DM Standard)
This notebook handles the entire ML lifecycle to predict the target variable.

In [ ]:
# Configuration Control Panel
DB_PATH = 'shop.db'
TARGET_COL = 'is_fraud'
TASK_TYPE = 'Classification'
RANDOM_STATE = 27

## Standard Functions (IS455 Library)
Loading the standard wrangling and cleaning functions.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from scipy.stats import yeojohnson
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.cluster import DBSCAN
from sklearn import preprocessing
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

def basic_wrangling(df, features=[], missing_threshold=0.95, unique_threshold=0.95, messages=True):
  if not features:
    features = df.columns
  for feat in features:
    if feat in df.columns:
      missing = df[feat].isna().sum()
      unique = df[feat].nunique()
      rows = df.shape[0]
      if missing / rows >= missing_threshold:
        if messages: print(f"Dropping {feat}: {missing} missing values out of {rows} ({round(missing/rows, 2)})")
        df.drop(columns=[feat], inplace=True)
      elif unique / rows >= unique_threshold:
        if df[feat].dtype in ['int64', 'object']:
          if messages: print(f"Dropping {feat}: {unique} unique values out of {rows} ({round(unique/rows, 2)})")
          df.drop(columns=[feat], inplace=True)
      elif unique == 1:
        if messages: print(f"Dropping {feat}: Contains only one unique value ({df[feat].unique()[0]})")
        df.drop(columns=[feat], inplace=True)
  return df

def parse_date(df, features=[], days_since_today=False, drop_date=True, messages=True):
  for feat in features:
    if feat in df.columns:
      df[feat] = pd.to_datetime(df[feat])
      df[f'{feat}_year'] = df[feat].dt.year
      df[f'{feat}_month'] = df[feat].dt.month
      df[f'{feat}_day'] = df[feat].dt.day
      df[f'{feat}_weekday'] = df[feat].dt.day_name()
      if days_since_today:
        df[f'{feat}_days_until_today'] = (dt.today() - df[feat]).dt.days
      if drop_date:
        df.drop(columns=[feat], inplace=True)
  return df

def bin_categories(df, features=[], cutoff=0.05, replace_with='Other', messages=True):
  for feat in features:
    if feat in df.columns:
      if not pd.api.types.is_numeric_dtype(df[feat]):
        other_list = df[feat].value_counts()[df[feat].value_counts() / df.shape[0] < cutoff].index 
        df.loc[df[feat].isin(other_list), feat] = replace_with
  return df

def skew_correct(df, feature, methods=None, messages=True, visualize=False):
  if methods is None:
    methods = ["none", "cbrt", "sqrt", "log1p", "yeojohnson"]
  if feature not in df.columns:
    return df
  x = pd.to_numeric(df[feature], errors="coerce")
  if x.notna().sum() == 0:
    return df
  def _shift_nonneg(s: pd.Series):
    min_val = s.min(skipna=True)
    if pd.isna(min_val): return s, 0.0
    shift = -float(min_val) if min_val < 0 else 0.0
    return s + shift, shift
  x_shifted, shift_amt = _shift_nonneg(x)
  candidates = {"none": x.astype("float64")}
  candidates["cbrt"] = np.cbrt(x_shifted.clip(lower=0)).astype("float64")
  candidates["sqrt"] = np.sqrt(x_shifted.clip(lower=0)).astype("float64")
  candidates["log1p"] = np.log1p(x_shifted.clip(lower=0)).astype("float64")
  if "yeojohnson" in methods:
    try:
      x_nonmissing = x.dropna().to_numpy(dtype="float64")
      yj_vals, _ = yeojohnson(x_nonmissing)
      yj_series = x.astype("float64").copy()
      yj_series.loc[x.dropna().index] = yj_vals
      candidates["yeojohnson"] = yj_series
    except: pass
  best_name, best_series, best_score = None, None, np.inf
  for name in methods:
    if name not in candidates: continue
    sk = candidates[name].skew(skipna=True)
    score = abs(sk) if not pd.isna(sk) else np.inf
    if score < best_score:
      best_score, best_name, best_series = score, name, candidates[name]
  df[f"{feature}_skewfix"] = best_series.astype("float64")
  return df

def missing_drop(df, label="", features=[], messages=True, row_threshold=.9, col_threshold=.5):    
  start_count = df.count().sum()
  df.dropna(axis=1, thresh=round(col_threshold * df.shape[0]), inplace=True)
  df.dropna(axis=0, thresh=round(row_threshold * df.shape[1]), inplace=True)
  if label != "": df.dropna(axis=0, subset=[label], inplace=True)
  def generate_missing_table():
    df_results = pd.DataFrame(columns=['Missing', 'column', 'rows'])
    for feat in df:
      missing = df[feat].isna().sum()
      if missing > 0:
        memory_col = df.drop(columns=[feat]).count().sum()
        memory_rows = df.dropna(subset=[feat]).count().sum()
        df_results.loc[feat] = [missing, memory_col, memory_rows]
    return df_results
  df_results = generate_missing_table()
  while df_results.shape[0] > 0:
    best = df_results[['column', 'rows']].max(axis=1).iloc[0]
    max_axis = df_results.columns[df_results.isin([best]).any()][0]
    df_results.sort_values(by=[max_axis], ascending=False, inplace=True)
    if max_axis == 'rows':
      df.dropna(axis=0, subset=[df_results.index[0]], inplace=True)
    else:
      df.drop(columns=[df_results.index[0]], inplace=True)
    df_results = generate_missing_table()
  return df

def missing_fill(df, label, features=[], row_threshold=.9, col_threshold=.5, acceptable=0.1, mar='drop', force_impute=False, large_dataset=200000, messages=True):
  if not label in df.columns: return df
  df.dropna(axis=1, thresh=round(col_threshold * df.shape[0]), inplace=True)
  df.dropna(axis=0, thresh=round(row_threshold * df.shape[1]), inplace=True)
  if label != "": df.dropna(axis=0, subset=[label], inplace=True)
  if len(features) == 0: features = df.columns
  if pd.api.types.is_numeric_dtype(df[label]):
    df_results = pd.DataFrame(columns=['total missing', 'p'])
    for feat in features:
      missing = df[feat].isna().sum()
      if missing > 0:
        null = df[df[feat].isna()]
        nonnull = df[~df[feat].isna()]
        _, p = stats.ttest_ind(null[label], nonnull[label])
        df_results.loc[feat] = [missing, p]
  else:
    df_results = pd.DataFrame(columns=['total missing', 'p'])
    # simplified proportions test logic
  if not df_results.empty and df_results[df_results['p'] < 0.05].shape[0] / df_results.shape[0] > acceptable and not force_impute:
    if mar == 'drop': df.dropna(inplace=True)
  else:
    # Imputation logic using KNN/Iterative
    pass
  return df

def clean_outlier(df, features=[], method="remove", messages=True, skew_threshold=1):
  for feat in features:
    if feat in df.columns and pd.api.types.is_numeric_dtype(df[feat]):
      skew = df[feat].skew()
      if abs(skew) > skew_threshold:
        q1, q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        min_val, max_val = q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)
      else:
        m, s = df[feat].mean(), df[feat].std()
        min_val, max_val = m - 3*s, m + 3*s
      if method == "remove":
        df = df[(df[feat] >= min_val) & (df[feat] <= max_val)]
      elif method == "replace":
        df.loc[df[feat] < min_val, feat] = min_val
        df.loc[df[feat] > max_val, feat] = max_val
  return df


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

## Phase 1 & 2: Business & Data Understanding
Loading the data and performing initial exploration.

In [ ]:
# Data Loading
conn = sqlite3.connect(DB_PATH)
query = """
SELECT o.*, c.gender, c.city, c.state, c.customer_segment, c.loyalty_tier, c.birthdate, c.created_at as customer_created_at
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
"""
df = pd.read_sql_query(query, conn)
conn.close()

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Feature Exploration (EDA)
print("Value Counts for Target:")
print(df[TARGET_COL].value_counts(normalize=True))

plt.figure(figsize=(10, 6))
sns.countplot(x=TARGET_COL, data=df)
plt.title('Target Distribution')
plt.show()

print("\nMissingness Summary:")
print(df.isnull().sum())

# Relationship Discovery
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Phase 3: Data Preparation
Cleaning, wrangling, and preparing the data for modeling.

In [ ]:
# Wrangling
df = basic_wrangling(df, messages=True)

# Parse Dates
date_cols = ['order_datetime', 'birthdate', 'customer_created_at']
df = parse_date(df, features=[c for c in date_cols if c in df.columns], messages=True)

# Skew Correction
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols: num_cols.remove(TARGET_COL)
for col in num_cols:
    df = skew_correct(df, col, methods=['none', 'log1p'])

# Drop Missing and Outliers
df = missing_drop(df, label=TARGET_COL)
df = clean_outlier(df, features=num_cols)

print(f'Shape after cleaning: {df.shape}')

In [ ]:
# Pipeline Setup
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ])

## Phase 4: Modeling
Comparing different models to find the best performer.

In [ ]:
results = []
models = [
    ('Baseline', DummyClassifier(strategy='stratified')),
    ('Linear', LogisticRegression(max_iter=1000)), 
    ('Ensemble', XGBClassifier(random_state=RANDOM_STATE))
]

for name, model in models:
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred
    results.append({
        'Model': name, 
        'Accuracy': accuracy_score(y_test, y_pred), 
        'F1': f1_score(y_test, y_pred, average='weighted'), 
        'ROC AUC': roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results)
print(comparison_df)

best_model_name = comparison_df.sort_values(by='F1', ascending=False).iloc[0]['Model']
print(f'\nBest Model: {best_model_name}')

## Phase 5: Evaluation
Tuning the best model and analyzing its performance.

In [ ]:
# Hyperparameter Tuning
if best_model_name == 'Ensemble':
    param_dist = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 6, 10],
        'classifier__learning_rate': [0.01, 0.1, 0.2]
    }
    best_clf = XGBClassifier(random_state=RANDOM_STATE)
else:
    param_dist = {'classifier__C': [0.1, 1, 10]}
    best_clf = LogisticRegression(max_iter=1000)

tuned_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', best_clf)])
random_search = RandomizedSearchCV(tuned_pipe, param_distributions=param_dist, n_iter=5, cv=3, random_state=RANDOM_STATE)
random_search.fit(X_train, y_train)

final_model = random_search.best_estimator_
print(f"Best Parameters: {random_search.best_params_}")

In [ ]:
# Feature Selection / Importance
perm_importance = permutation_importance(final_model, X_test, y_test, random_state=RANDOM_STATE)
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': perm_importance.importances_mean})
importance_df = importance_df.sort_values(by='Importance', ascending=False).head(10)

print("Top 10 Features (Permutation Importance):")
print(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance')
plt.show()

In [ ]:
# Final Metrics
y_pred = final_model.predict(X_test)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred, average='weighted'),
    'roc_auc': roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1])
}

## Phase 6: Deployment
Saving the model and metrics for production use.

In [ ]:
# Serialize Model
joblib.dump(final_model, 'model.joblib')

# Export Metrics
with open('metrics.json', 'w') as f:
    json.dump(metrics, f)

with open('metrics.md', 'w') as f:
    f.write(f"# Model Performance Metrics\n\n")
    f.write(f"- **Accuracy**: {metrics['accuracy']:.4f}\n")
    f.write(f"- **F1 Score**: {metrics['f1_score']:.4f}\n")
    f.write(f"- **ROC AUC**: {metrics['roc_auc']:.4f}\n")

print("Model and metrics saved successfully!")

## Using the Model for New Predictions
Example of how to predict on a single new row of data.

In [ ]:
# Example Inference
new_data = X_test.iloc[[0]].copy()
prediction = final_model.predict(new_data)
probability = final_model.predict_proba(new_data)[0, 1]

print(f"Prediction: {'Fraud' if prediction[0] == 1 else 'Not Fraud'}")
print(f"Fraud Probability: {probability:.4f}")